# ds004752 V5.6 Tranche 2.2 Feature Matrix Plan

Purpose: record a claim-closed feature-matrix plan after split registry lock and feature provenance pass.

Integrity boundary: this notebook does not materialize feature values, does not train models, does not run comparators, does not compute efficacy metrics, and does not open claims.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update repo

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)

## 3. Confirm Tranche 2.2 code is present

In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -8

from pathlib import Path

required_files = [
    Path('src/v56/feature_matrix_plan.py'),
    Path('configs/v56/feature_matrix_plan.json'),
    Path('tests/unit/test_v56_feature_matrix_plan.py'),
    Path('bootstrap/run_v56_feature_matrix_plan.sh'),
    Path('docs/24_v56_feature_matrix_plan_runbook_2026-05-04.md'),
]
missing = [str(p) for p in required_files if not p.exists()]
print({'missing_required_files': missing})
assert not missing, missing

cli_text = Path('src/cli.py').read_text(encoding='utf-8')
assert 'v56-feature-matrix-plan' in cli_text
assert 'run_v56_feature_matrix_plan' in cli_text
print('V5.6 Tranche 2.2 feature-matrix plan code is present.')

## 4. Configure source-of-truth paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260424T100159866284Z'
SPLIT_LOCK_RUN = DRIVE_ROOT / 'artifacts/v56_split_registry_lock/latest.txt'
FEATURE_PROVENANCE_RUN = DRIVE_ROOT / 'artifacts/v56_feature_provenance_populated/latest.txt'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/v56_feature_matrix_plan'

for path in [GATE0_RUN, SPLIT_LOCK_RUN, FEATURE_PROVENANCE_RUN]:
    print(path, path.exists())
    assert path.exists(), f'Missing required path: {path}'
print('OUTPUT_ROOT =', OUTPUT_ROOT)

## 5. Preflight upstream Tranche 2.1 artifacts

In [ ]:
import json

split_dir = Path(SPLIT_LOCK_RUN.read_text().strip())
feature_dir = Path(FEATURE_PROVENANCE_RUN.read_text().strip())
split_lock = json.loads((split_dir / 'v56_split_registry_lock.json').read_text())
feature = json.loads((feature_dir / 'v56_feature_provenance_populated.json').read_text())

preflight = {
    'split_dir': str(split_dir),
    'feature_dir': str(feature_dir),
    'split_status': split_lock.get('status'),
    'split_claim_closed': split_lock.get('claim_closed'),
    'n_folds': len(split_lock.get('folds', [])),
    'test_time_inference': split_lock.get('test_time_inference'),
    'feature_status': feature.get('status'),
    'feature_claim_closed': feature.get('claim_closed'),
    'required_links_satisfied': feature.get('required_links_satisfied'),
    'missing_sources': feature.get('missing_sources'),
}
print(json.dumps(preflight, indent=2))

assert preflight['split_status'] == 'locked_subject_level_split_registry', preflight
assert preflight['split_claim_closed'] is True, preflight
assert preflight['n_folds'] == 30, preflight
assert preflight['test_time_inference']['modality'] == 'scalp_eeg_only', preflight
assert preflight['test_time_inference']['allow_ieeg'] is False, preflight
assert preflight['test_time_inference']['allow_beamforming_bridge'] is False, preflight
assert preflight['feature_status'] == 'populated_source_hashes_and_split_links', preflight
assert preflight['feature_claim_closed'] is True, preflight
assert all(preflight['required_links_satisfied'].values()), preflight
assert preflight['missing_sources'] == [], preflight
print('Preflight passed: Tranche 2.1 artifacts are ready.')

## 6. Run V5.6 Tranche 2.2 feature-matrix plan

In [ ]:
%cd /content/eeg-ds004752
!bash bootstrap/run_v56_feature_matrix_plan.sh {GATE0_RUN} {SPLIT_LOCK_RUN} {FEATURE_PROVENANCE_RUN} {OUTPUT_ROOT}

## 7. Load latest plan artifact

In [ ]:
latest = OUTPUT_ROOT / 'latest.txt'
assert latest.exists(), latest
run_dir = Path(latest.read_text().strip())
assert run_dir.exists(), run_dir

plan = json.loads((run_dir / 'v56_feature_matrix_plan.json').read_text())
summary = json.loads((run_dir / 'v56_feature_matrix_plan_summary.json').read_text())
validation = json.loads((run_dir / 'v56_feature_matrix_plan_validation.json').read_text())

compact = {
    'run_dir': str(run_dir),
    'status': plan.get('status'),
    'validation_status': validation.get('status'),
    'claim_closed': plan.get('claim_closed'),
    'claim_ready': plan.get('claim_ready'),
    'n_primary_feature_sets': len(plan.get('primary_feature_sets', [])),
    'n_privileged_train_time_sources': len(plan.get('privileged_train_time_sources', [])),
    'feature_matrix_materialized': plan.get('scientific_boundary', {}).get('feature_matrix_materialized'),
    'model_training_run': plan.get('scientific_boundary', {}).get('model_training_run'),
    'efficacy_metrics_computed': plan.get('scientific_boundary', {}).get('efficacy_metrics_computed'),
}
print(json.dumps(compact, indent=2))

## 8. Integrity assertions

In [ ]:
assert plan['status'] == 'planned_feature_matrix_contract_recorded', plan
assert validation['status'] == 'v56_feature_matrix_plan_validation_passed', validation
assert validation['blocking_errors'] == [], validation
assert plan['claim_closed'] is True, plan
assert plan['claim_ready'] is False, plan
assert plan['test_time_inference']['modality'] == 'scalp_eeg_only', plan
assert plan['test_time_inference']['allow_ieeg'] is False, plan
assert plan['test_time_inference']['allow_beamforming_bridge'] is False, plan
assert plan['scientific_boundary']['feature_matrix_materialized'] is False, plan
assert plan['scientific_boundary']['model_training_run'] is False, plan
assert plan['scientific_boundary']['comparator_execution_run'] is False, plan
assert plan['scientific_boundary']['efficacy_metrics_computed'] is False, plan
assert summary['claim_closed'] is True, summary
assert summary['feature_matrix_materialized'] is False, summary
assert summary['model_training_run'] is False, summary
assert summary['efficacy_metrics_computed'] is False, summary

for feature_set in plan['primary_feature_sets']:
    if feature_set['allowed_at_test_time']:
        assert feature_set['source_modality'] == 'scalp_eeg', feature_set
for source in plan['privileged_train_time_sources']:
    assert source['allowed_at_test_time'] is False, source

print('V5.6 Tranche 2.2 feature-matrix plan passed integrity assertions.')

## 9. Decision gate closeout

In [ ]:
closeout = {
    'tranche2_2_status': 'feature_matrix_plan_recorded',
    'run_dir': str(run_dir),
    'claim_closed': True,
    'claim_ready': False,
    'feature_matrix_materialized': False,
    'model_training_run': False,
    'comparator_execution_run': False,
    'efficacy_metrics_computed': False,
    'next_step': 'manual_review_then_implement_feature_matrix_materializer_or_leakage_plan_only_if_plan_passes',
    'not_allowed_next': [
        'do_not_train_RIFT_Net_Lite_yet',
        'do_not_run_A3_A4_comparators_yet',
        'do_not_open_efficacy_claim',
    ],
}
print(json.dumps(closeout, indent=2))
print('\n=== report preview ===')
print((run_dir / 'v56_feature_matrix_plan_report.md').read_text()[:2500])